In [ ]:
import gdsfactory as gf
import numpy as np
import math

gf.gpdk.PDK.activate()

In [ ]:
def n_SiN (wavelength):

    return np.sqrt(1+(2.9144*wavelength**2)/(wavelength**2-0.1366**2)+(0.004873)/(wavelength**2-1.6606**2))


def n_SiO2 (wavelength):

    return np.sqrt(1+(1.1056*wavelength**2)/(wavelength**2-0.078**2)+(2.360*wavelength**2)/(wavelength**2-16.681**2)) + 0.002


def phase_match_calc(wavelength,FSR,ng,neff):
    R = 1000 * (wavelength)**2 / (2*math.pi*FSR*ng)  ##um
    m = neff*2*math.pi*R/wavelength
    new_R = round(m) * wavelength/(neff*2*math.pi)
    new_FSR = 1000 * (wavelength)**2 / (2*math.pi*new_R*ng)

    return new_R, new_FSR



In [ ]:
print(f"core = {n_SiN(1.310)}")
print(f"cladd = {n_SiO2(1.310)}")

In [2]:
def Ring_sensor_spec( wavelength = 1.310,

                      wg_width = 1.0,

                      ring_sens_radius = 100,
                      ng_ring_sens = 1.855907, # Group index of Silicon and
                      gap_ring_sens = 0.15,

                      ring1_radius = 100,
                      ng_ring1 = 1.855907,
                      gap_ring1 = 0.15,

                      ring2_radius = 100,
                      ng_ring2 = 1.855907,
                      gap_ring2 = 0.15,

                      ring3_radius = 100,
                      ng_ring3 = 1.855907,
                      gap_ring3 = 0.15,

                      ring4_radius = 100,
                      ng_ring4 = 1.855907,
                      gap_ring4 = 0.15,

                      ring5_radius = 100,
                      ng_ring5 = 1.855907,
                      gap_ring5 = 0.15,

                      ring6_radius = 100,
                      ng_ring6 = 1.855907,
                      gap_ring6 = 0.15,

                      wg_separation = 2000,
                      spec_wg_angle = -45,

                      wg_multimode_width = 1000,
                      wg_multimode_length = 2000,
                      neff= 1.574385, # for 1.2 um width, 220nm thick at 1310nm SiN
                      FSR = 20,
                      strip_length = 400,
                      Ring_sense_input_pos = (0,0),
                      thickness = 0.22,
                      bend_radius = 20,
                      taper_length = 10,
                      layer = (733,727),):

    ## Cálculo del delta de L para cada ring por separado

    newR_sensor, newFSR_sensor = phase_match_calc(wavelength,FSR,ng_ring_sens,neff)

    newR1, newFSR_R1 = phase_match_calc(wavelength,FSR,ng_ring1,neff)

    newR2, newFSR_R2 = phase_match_calc(wavelength,FSR,ng_ring2,neff)

    newR3, newFSR_R3 = phase_match_calc(wavelength,FSR,ng_ring3,neff)

    newR4, newFSR_R4 = phase_match_calc(wavelength,FSR,ng_ring4,neff)

    newR5, newFSR_R5 = phase_match_calc(wavelength,FSR,ng_ring5,neff)

    newR6, newFSR_R6 = phase_match_calc(wavelength,FSR,ng_ring6,neff)


    # //////  INICIO DE LA FUNCIÓN
    # We define an sketch where we will place the components
    c = gf.Component()


    ############## MM In ########################

    cross_MM_in_out = gf.cross_section.strip(
    width=wg_multimode_width,
    layer=layer
    )

    MM_in = gf.components.straight(length=wg_multimode_length,cross_section=cross_MM_in_out)

    MM_in_ref = c.add_ref(MM_in)
    MM_in_ref.move((Ring_sense_input_pos[0],Ring_sense_input_pos[1]))

    ###############################################

    ############## Taper In ######################

    cross_taper_in = gf.cross_section.strip(
    width=wg_multimode_width,
    layer=layer
    )
    cross_taper_out = gf.cross_section.strip(
    width=wg_width,
    layer=layer
    )

    taper_in = gf.components.taper_cross_section(length=taper_length,cross_section1=cross_taper_in,cross_section2=cross_taper_out)
    taper_in_ref = c.add_ref(taper_in)
    taper_in_ref.connect("o1", MM_in_ref.ports["o2"])

    ##############################################

    ############## STRIP ########################

    cross_strip_wg = gf.cross_section.strip(
    width=wg_width,
    layer=layer
    )

    strip_wg = gf.components.straight(length=,cross_section=cross_MM_in_out)

    MM_in_ref = c.add_ref(MM_in)
    MM_in_ref.move((Ring_sense_input_pos[0],Ring_sense_input_pos[1]))

    ###############################################




    ############## Bend_1_input ###################

    bend_out = gf.components.bend_euler(width=wg_width,radius=bend_radius,layer=layer)
    ref_bend_out = c.add_ref(bend_out)
    ref_bend_out.connect("o1", ref_bend_in.ports["o2"])

    ###########################################





    ############## BUS SiN ###################

    cross_strip_wg = gf.cross_section.strip(
    width=wg_width,
    layer=layer
    )

    STRP_SiN_in = gf.components.straight(length=strip_length,cross_section=cross_strip_wg)
    STRP_SiN_in_ref = c.add_ref(STRP_SiN_in)
    STRP_SiN_in_ref.move((RR_pos[0],RR_pos[1]))

    ###############################################


    ##############    Ring   #######################

    ring = gf.components.rings.ring(radius=new_R, width=wg_width_strp, angle_resolution=2.5, layer=layer, angle=360).copy()
    ring_ref = c.add_ref(ring)
    ring_ref.move((RR_pos[0] + strip_length/2 , RR_pos[1] + new_R + wg_width_strp + gap ))

    #################################################

    # # Create text
    # text = gf.components.text(
    #     text=f"AP_FSR_{new_FSR:.4f}nm_R_{new_R:.2f}um_Juanes",
    #     size=5,          # height in microns
    #     layer=layer
    # )
    #
    # # Add to layout
    # text_ref = c.add_ref(text)
    #
    # # Move it if needed
    # text_ref.move((RR_pos[0] + 20, RR_pos[1] + 20))



    total_x_length = abs(STRP_SiN_in_ref.ports["o1"].center[0] - STRP_SiN_in_ref.ports["o2"].center[0] ) ## um
    total_y_length = 3*wg_width_strp + gap + 2*new_R   ## um


    # ######BOX, OUT#######
    #
    # box_poly_out = gf.Component()
    #
    # box_poly_out.add_polygon([(0,13),(53,13),(53,-13),(0,-13)], layer= (1328,1609))
    # box_out_ref = c.add_ref(box_poly_out)
    #
    # box_out_ref.move((RR_pos[0] + total_x_length,RR_pos[1]))
    #
    #
    # ###################



    return c, total_x_length,total_y_length

RR_scketch = gf.Component()

y_pos = 0

for fsr in    [2.0]:
    # [6.0,7.0,8.0,9.0,9.5,10,10.2,10.5]:
    RR = All_pass_ring(gap = 0.500 ,FSR = fsr ,RR_pos=(0,y_pos))
    RR_scketch.add_ref(RR[0])
    y_pos += RR[2] + 20


RR_scketch.draw_ports()
RR_scketch.plot()
RR_scketch.write("RR_SiN_Juanes_1.gds")
RR_scketch.show()

SyntaxError: duplicate argument 'gap_ring_sens' in function definition (869579312.py, line 14)